<a href="https://colab.research.google.com/github/haniyathul/TESIS-HANIY/blob/main/SIMULASI%20PEMBENTUKAN%20ESF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================================================
# ================================================================
# PEMBENTUKAN BASIS EIGENVECTOR SPATIAL FILTERING (ESF)
#
# INPUT:
# Matriks W hasil pembangkitan data simulasi
#
# OUTPUT PER GRID:
# 1. ESF_BASIS_GRIDxx_KNN5.npz
# 2. ESF_BASIS_GRIDxx_DISTANCE.npz
# 3. ESF_BASIS_GRIDxx_IDW.npz
#
# TOTAL AKHIR:
# 5 GRID × 3 W = 15 FILE ESF
# ================================================================
# ================================================================

import gc
import numpy as np
import pandas as pd

import scipy.sparse as sp

from pathlib import Path
from scipy.linalg import eigh


# ================================================================
# 1. FOLDER OUTPUT
# ================================================================

ESF_OUTPUT_DIR = Path(
    "/content/ESF_BASIS"
)

ESF_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 2. METODE W
# ================================================================

W_METHODS = [
    "KNN5",
    "DISTANCE",
    "IDW"
]


# ================================================================
# 3. FUNGSI MORAN'S I UNTUK EIGENVECTOR
# ================================================================

def hitung_moran_eigenvector(
    eigenvectors,
    W
):

    n = W.shape[0]

    S0 = float(
        W.sum()
    )


    W_E = W @ eigenvectors


    numerator = np.sum(
        eigenvectors * W_E,
        axis=0
    )


    denominator = np.sum(
        eigenvectors ** 2,
        axis=0
    )


    moran = (

        (n / S0)

        *

        (
            numerator
            /
            denominator
        )

    )


    return np.asarray(
        moran,
        dtype=float
    )


# ================================================================
# 4. FUNGSI MEMBENTUK SATU BASIS ESF
# ================================================================

def bentuk_satu_esf(
    grid_size,
    w_method
):

    n = (
        grid_size
        *
        grid_size
    )


    # ------------------------------------------------------------
    # FILE W
    # ------------------------------------------------------------

    W_FILE = Path(

        f"/content/"
        f"W_GRID{grid_size}_{w_method}.npz"

    )


    if not W_FILE.exists():

        raise FileNotFoundError(

            f"""
FILE TIDAK DITEMUKAN:

{W_FILE.name}

Upload dahulu file:
W_GRID{grid_size}_{w_method}.npz
ke folder /content.
"""
        )


    print("\n")
    print("=" * 100)

    print(
        f"GRID {grid_size}×{grid_size}"
        f" | W = {w_method}"
    )

    print("=" * 100)


    # ============================================================
    # 5. LOAD W
    # ============================================================

    W = sp.load_npz(
        W_FILE
    ).tocsr()


    assert W.shape == (
        n,
        n
    ), (
        f"Ukuran W salah: {W.shape}, "
        f"seharusnya {(n,n)}"
    )


    print(
        "Ukuran W :",
        W.shape
    )


    # ============================================================
    # 6. SIMETRISASI
    #
    # C = (W + W') / 2
    #
    # Digunakan agar operator ESF simetris sehingga
    # eigenvalue dan eigenvector real.
    # ============================================================

    C = (

        W
        +
        W.T

    ) * 0.5


    C = C.tocsr()


    C.setdiag(
        0
    )

    C.eliminate_zeros()


    # ============================================================
    # 7. PEMBENTUKAN OPERATOR ESF
    #
    # M = I - 11'/n
    #
    # Omega = M C M
    #
    # Di bawah ini dilakukan dengan double centering
    # tanpa membentuk M secara eksplisit.
    # ============================================================

    C_dense = C.toarray()


    row_mean = C_dense.mean(
        axis=1,
        keepdims=True
    )


    col_mean = C_dense.mean(
        axis=0,
        keepdims=True
    )


    grand_mean = C_dense.mean()


    OMEGA = (

        C_dense

        -

        row_mean

        -

        col_mean

        +

        grand_mean

    )


    # ------------------------------------------------------------
    # Jaga simetri numerik
    # ------------------------------------------------------------

    OMEGA = (

        OMEGA

        +

        OMEGA.T

    ) / 2


    print(
        "Operator MCM : SELESAI"
    )


    # ============================================================
    # 8. EIGENDECOMPOSITION
    # ============================================================

    eigenvalues, eigenvectors = eigh(

        OMEGA,

        overwrite_a=True,

        check_finite=False

    )


    # ============================================================
    # 9. URUTKAN EIGENVALUE TERBESAR → TERKECIL
    # ============================================================

    urutan = np.argsort(
        eigenvalues
    )[::-1]


    eigenvalues = (
        eigenvalues[
            urutan
        ]
    )


    eigenvectors = (
        eigenvectors[
            :,
            urutan
        ]
    )


    # ============================================================
    # 10. HAPUS EIGENVALUE NUMERIK ~ 0
    #
    # Komponen konstan dari centering tidak digunakan.
    # ============================================================

    tolerance = (

        1e-10

        *

        max(
            1.0,
            np.max(
                np.abs(
                    eigenvalues
                )
            )
        )

    )


    keep = (

        np.abs(
            eigenvalues
        )

        >

        tolerance

    )


    eigenvalues = (
        eigenvalues[
            keep
        ]
    )


    eigenvectors = (
        eigenvectors[
            :,
            keep
        ]
    )


    # ============================================================
    # 11. ORIENTASI TANDA EIGENVECTOR
    #
    # Secara matematis E dan -E sama.
    # Orientasi dibuat konsisten untuk reproducibility.
    # ============================================================

    for j in range(
        eigenvectors.shape[1]
    ):

        idx = np.argmax(

            np.abs(
                eigenvectors[
                    :,
                    j
                ]
            )

        )


        if (
            eigenvectors[
                idx,
                j
            ]
            <
            0
        ):

            eigenvectors[
                :,
                j
            ] *= -1


    # ============================================================
    # 12. MORAN'S I SETIAP EIGENVECTOR
    #
    # Menggunakan W asli.
    # ============================================================

    moran_I = hitung_moran_eigenvector(

        eigenvectors,

        W

    )


    # ============================================================
    # 13. IDENTIFIKASI EIGENVALUE POSITIF
    #
    # Ini akan berguna nanti pada tahap seleksi ESF.
    # ============================================================

    positive_mask = (

        eigenvalues > 0

    )


    positive_index = np.where(

        positive_mask

    )[0]


    n_positive = int(
        positive_mask.sum()
    )


    n_negative = int(

        (
            eigenvalues < 0
        ).sum()

    )


    print(
        "Jumlah lokasi          :",
        n
    )

    print(
        "Jumlah eigenvector     :",
        eigenvectors.shape[1]
    )

    print(
        "Eigenvalue positif     :",
        n_positive
    )

    print(
        "Eigenvalue negatif     :",
        n_negative
    )

    print(
        "Moran I minimum       :",
        round(
            float(
                np.min(
                    moran_I
                )
            ),
            6
        )
    )

    print(
        "Moran I maksimum      :",
        round(
            float(
                np.max(
                    moran_I
                )
            ),
            6
        )
    )


    # ============================================================
    # 14. SAVE FILE ESF
    # ============================================================

    OUTPUT_FILE = (

        ESF_OUTPUT_DIR

        /

        (
            f"ESF_BASIS_GRID"
            f"{grid_size}_"
            f"{w_method}.npz"
        )

    )


    np.savez_compressed(

        OUTPUT_FILE,

        grid_size=
            np.array(
                grid_size,
                dtype=np.int32
            ),

        n=
            np.array(
                n,
                dtype=np.int32
            ),

        w_method=
            np.array(
                w_method
            ),

        eigenvalues=
            eigenvalues.astype(
                np.float32
            ),

        eigenvectors=
            eigenvectors.astype(
                np.float32
            ),

        moran_I=
            moran_I.astype(
                np.float32
            ),

        positive_mask=
            positive_mask.astype(
                np.int8
            ),

        positive_index=
            positive_index.astype(
                np.int32
            )

    )


    print(
        "\n✅ FILE BERHASIL:",
        OUTPUT_FILE.name
    )


    # ============================================================
    # 15. BERSIHKAN MEMORY
    # ============================================================

    hasil = {

        "Grid":
            f"{grid_size}x{grid_size}",

        "W_Method":
            w_method,

        "N":
            n,

        "Jumlah_Eigenvector":
            eigenvectors.shape[1],

        "Eigen_Positif":
            n_positive,

        "Eigen_Negatif":
            n_negative,

        "Moran_Min":
            float(
                moran_I.min()
            ),

        "Moran_Max":
            float(
                moran_I.max()
            ),

        "File":
            OUTPUT_FILE.name

    }


    del W
    del C
    del C_dense
    del OMEGA
    del eigenvectors

    gc.collect()


    return hasil


# ================================================================
# 16. FUNGSI PER GRID
#
# 1 GRID → 3 FILE ESF
# ================================================================

def bentuk_esf_grid(
    grid_size
):

    print("\n")
    print("#" * 100)

    print(
        f"PEMBENTUKAN ESF "
        f"GRID {grid_size}×{grid_size}"
    )

    print("#" * 100)


    hasil_grid = []


    for w_method in W_METHODS:

        hasil = bentuk_satu_esf(

            grid_size=
                grid_size,

            w_method=
                w_method

        )


        hasil_grid.append(
            hasil
        )


    summary = pd.DataFrame(
        hasil_grid
    )


    print("\n")
    print("=" * 100)

    print(
        f"SUMMARY GRID "
        f"{grid_size}×{grid_size}"
    )

    print("=" * 100)


    print(

        summary.to_string(
            index=False
        )

    )


    return summary


print(
    "✅ FUNGSI PEMBENTUKAN ESF SIAP"
)

✅ FUNGSI PEMBENTUKAN ESF SIAP


In [2]:
ESF_GRID10 = bentuk_esf_grid(
    10
)



####################################################################################################
PEMBENTUKAN ESF GRID 10×10
####################################################################################################


FileNotFoundError: 
FILE TIDAK DITEMUKAN:

W_GRID10_KNN5.npz

Upload dahulu file:
W_GRID10_KNN5.npz
ke folder /content.
